Copyright 2026 Google LLC
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

      http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License

First setup an environment to enable one to learn how to use Highway.

In [ ]:
!apt update -qq;
!apt-get update -qq;
!apt update
!apt install cmake gcc g++ git libatomic1 libgtest-dev ninja-build
!apt-get upgrade

Now get the highway code and install Highway. This may take about 10 minutes.  To not extend build time further, do not build tests and examples.

In [ ]:
#!git clone https://github.com/google/highway
%cd highway
!mkdir build
%cd build
!cmake -GNinja \
       -DHWY_ENABLE_TESTS=OFF \
       -DHWY_ENABLE_EXAMPLES=OFF \
       -DHWY_ENABLE_CONTRIB=ON \
       -DCMAKE_INSTALL_PREFIX=/content/highway_install ..
!ninja
!ninja install
%cd ..
%cd ..

As a first example, let us try out a program to do a vector addition.

In [ ]:
%%writefile vector_add.cpp
#include <stddef.h>
#include <stdint.h>

#include <chrono>
#include <iostream>
#include <numeric>
#include <vector>

float SumArray(const float* array, size_t count) {
  float sum = 0;
  for (size_t i = 0 ; i < count; i++) {
      sum += array[i];
  }
  return sum;
}

int main() {
  const size_t count = 1025;
  std::vector<float> data(count);
  std::iota(data.begin(), data.end(), 1.0f);
  //start timer
  auto start = std::chrono::high_resolution_clock::now();
  float sum = SumArray(data.data(), count);
  //stop timer and print time and sum
  auto stop = std::chrono::high_resolution_clock::now();
  std::chrono::duration<double, std::milli>  diff = stop - start;
  std::cout << "Execution time: " << diff.count() << " ms" << std::endl;
  std::cout << "Sum: " << sum << std::endl;

  return 0;
}

Now compile and run this program with no optimization.

In [ ]:
!g++ -std=c++17 -O0 -o vector_add.o -c vector_add.cpp
!g++ -o vectoradd vector_add.o
!./vector_add

Now compile and run an optimized executable.

In [ ]:
!g++ -O3 vector_add.cpp -o vector_add
!./vector_add

Now use Highway to do explicit vectorization.

In [ ]:
%%writefile vector_add_hwy.cpp
#include <stddef.h>
#include <stdint.h>

#include <chrono>
#include <iostream>
#include <numeric>
#include <vector>


// >>>> for dynamic dispatch only, skip if you want static dispatch

// First undef to prevent error when re-included.
#undef HWY_TARGET_INCLUDE
// For dynamic dispatch, specify the name of the current file (unfortunately
// __FILE__ is not reliable) so that foreach_target.h can re-include it.
// The absolute path seems to be required
#define HWY_TARGET_INCLUDE "/content/vector_add_hwy.cpp"
// Generates code for each enabled target by re-including this source file.
#include "hwy/foreach_target.h"  // IWYU pragma: keep

// <<<< end of dynamic dispatch
//
#include "hwy/highway.h"

HWY_BEFORE_NAMESPACE();
namespace hwy {
namespace HWY_NAMESPACE {
namespace hn = hwy::HWY_NAMESPACE;

float SumArraySIMD(const float* HWY_RESTRICT array, size_t count) {
  const hn::ScalableTag<float> d;
  using V = hn::Vec<decltype(d)>;
  V sum = hn::Zero(d);
  size_t i = 0;
  const size_t N = hn::Lanes(d);
  if (count >= N) {
    for (; i <= count - N; i += N) {
      sum = hn::Add(sum, hn::LoadU(d, array + i));
    }
  }
  float total = hn::ReduceSum(d, sum);
  // Simple scalar remainder handling
  for (; i < count; ++i) {
    total += array[i];
  }
  return total;
}

}  // namespace HWY_NAMESPACE
}  // namespace hwy
HWY_AFTER_NAMESPACE();

#if HWY_ONCE
namespace hwy {
HWY_EXPORT(SumArraySIMD);

float CallSumArraySIMD(const float* array, size_t count) {
  return HWY_DYNAMIC_DISPATCH(SumArraySIMD)(array, count);
}

}  // namespace hwy

int main() {
  const size_t count = 1025;
  std::vector<float> data(count);
  std::iota(data.begin(), data.end(), 1.0f);
  //start timer
  auto start = std::chrono::high_resolution_clock::now();
  float sum = hwy::CallSumArraySIMD(data.data(), count);
  //stop timer and print time and sum
  auto stop = std::chrono::high_resolution_clock::now();
  std::chrono::duration<double, std::milli>  diff = stop - start;
  std::cout << "Execution time: " << diff.count() << " ms" << std::endl;
  std::cout << "Sum: " << sum << std::endl;

  return 0;
}
#endif  // HWY_ONCE

Compile and run explicitly vectorized version.

In [ ]:
!g++ -std=c++17 -O3 -I/content/highway_install/include -o vector_add_hwy.o -c vector_add_hwy.cpp
!g++ -o vector_add_hwy vector_add_hwy.o -L/content/highway_install/lib -lhwy_contrib  -lhwy
!./vector_add_hwy

Try increasing the array size, then recompile and run the examples again.  For what array size is the explicitly SIMD vectorized program faster than the program with O0 optimization?  What about for O3 optimization?

Ideally, make a plot showing execution time against array size for the program with O0 optimization, O3 optimization and explicitly vectorized using Highway.